In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, GPT2Config, TrainingArguments, Trainer
from sentencepiece import SentencePieceTrainer, SentencePieceProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")

PyTorch device: cpu


# Stage 1

## Load Dataset

In [2]:
# This does not work: "RuntimeError: Dataset scripts are no longer supported, but found ptb_text_only.py"
#dataset = load_dataset("ptb_text_only")

data_files = {
    "train": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.train.txt",
    "test": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.test.txt",
    "valid": "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb.valid.txt",
}
dataset = load_dataset("text", data_files=data_files)
del data_files

## Preprocess and Tokenize

In [3]:
# Join all rows in all splits by newlines
joined_text = "\n".join([row["text"] for split in dataset.values() for row in split])
print(f"Total characters: {len(joined_text)}")

if not os.path.isfile("corpus.txt"):
    with open("corpus.txt", "w") as f:
        f.write(joined_text)

# Train tokenizer if it is not there
if not os.path.isfile("tokenizer.model"):
    SentencePieceTrainer.train(
        input="corpus.txt",
        model_prefix="tokenizer",
        vocab_size=8000,
        character_coverage=1.0
    )

# Load and use tokenizer
tokenizer = SentencePieceProcessor()
tokenizer.load("tokenizer.model")
joined_tokens = tokenizer.encode(joined_text)

print(f"Total tokens: {len(joined_tokens)}")

del joined_tokens, joined_text

tokenized_dataset = {
    split: tokenizer.encode("\n".join([row["text"] for row in dataset[split]]))
    for split in ["train", "test", "valid"]
}

Total characters: 5951344
Total tokens: 1456873


## Initialize Model

In [4]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size(),
    n_positions=128,
    n_ctx=128,
    n_embd=256,
    n_layer=4,
    n_head=4,
    n_inner=1024,
    embd_pdrop=0.1,
    attn_pdrop=0.1,
    resid_pdrop=0.1,
    tie_word_embeddings=True,
    bos_token_id=tokenizer.bos_id(),
    eos_token_id=tokenizer.eos_id(),
    pad_token_id=tokenizer.pad_id(),
)
model = AutoModelForCausalLM.from_config(config)
model.to(device)

The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(8000, 256)
    (wpe): Embedding(128, 256)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-3): 4 x GPT2Block(
        (ln_1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=768, nx=256)
          (c_proj): Conv1D(nf=256, nx=256)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=1024, nx=256)
          (c_proj): Conv1D(nf=256, nx=1024)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=256, out_features=8000, bias=False)
)

# Train